In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time # Importação necessária para criar as pausas de segurança

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def webscraping(produto, parametro, chave):
    alvo = produto.find(parametro, class_=chave)
    return alvo.get_text().strip() if alvo else "N/A"

# 1. Definimos uma lista com as categorias e suas URLs. 
# O '{}' servirá como espaço reservado para inserir o número da página no loop.
categorias_alvo = [
    {
        "nome": "Feminino",
        "url": "https://www.dafiti.com.br/calcados-femininos/?page={}"
    },
    {
        "nome": "Masculino",
        "url": "https://www.dafiti.com.br/calcados-masculinos/?page={}"
    }
]

NUM_PAGINAS = 900 
todos_produtos = []

# 2. Loop externo: Percorre cada categoria (Feminino, depois Masculino)
for categoria in categorias_alvo:
    print(f"\n--- Iniciando extração: {categoria['nome']} ---")
    
    # 3. Loop interno: Percorre as páginas da categoria atual
    for pagina in range(1, NUM_PAGINAS + 1):
        url_paginada = categoria['url'].format(pagina)
        
        response = requests.get(url_paginada, headers=headers)
        print(f"[{categoria['nome']}] Página {pagina}: Status {response.status_code}")
        
        # Interrompe se a página não for encontrada (erro 404, por exemplo)
        if response.status_code != 200:
            print("Página não encontrada ou bloqueada. Indo para a próxima categoria.")
            break
            
        soup = BeautifulSoup(response.text, "html.parser")
        bloco_infos = soup.find_all("div", class_="product-box-detail")
        
        # Interrompe o loop das 900 páginas se chegar em uma página sem produtos
        if not bloco_infos:
            print(f"Fim dos produtos na categoria {categoria['nome']}.")
            break
            
        for bloco in bloco_infos:
            entrada = {
                "Categoria": categoria['nome'], # Adiciona a origem do dado
                "Marca": webscraping(bloco, "div", "product-box-brand"),
                "Cor": webscraping(bloco, "div", "product-box-colors"),
                "Vendedor": webscraping(bloco, "div", "product-box-seller"),
                "Produto": webscraping(bloco, "p", "product-box-title"),
                "Tags": webscraping(bloco, "span", "product-box-category-name"),
                "Preço Original": webscraping(bloco, "span", "product-box-price-from"),
                "Preço Pix": webscraping(bloco, "span", "product-box-price-to"),
                "Preço Parcelado": webscraping(bloco, "span", "product-box-installment-value"),
                "Quantidade de Parcelas": webscraping(bloco, "span", "product-box-installment-parcel")
            }
            todos_produtos.append(entrada)
        
        # Pausa de 1 segundo para não sobrecarregar o servidor e evitar banimento de IP
        time.sleep(1) 

print(f"\nTotal coletado: {len(todos_produtos)} produtos.")

dados = pd.DataFrame(todos_produtos)
dados.to_csv('202608_DadosDafiti_Combinados.csv', index=False)


--- Iniciando extração: Feminino ---
[Feminino] Página 1: Status 200
[Feminino] Página 2: Status 200
[Feminino] Página 3: Status 200
[Feminino] Página 4: Status 200
[Feminino] Página 5: Status 200
[Feminino] Página 6: Status 200
[Feminino] Página 7: Status 200
[Feminino] Página 8: Status 200
[Feminino] Página 9: Status 200
[Feminino] Página 10: Status 200
[Feminino] Página 11: Status 200
[Feminino] Página 12: Status 200
[Feminino] Página 13: Status 200
[Feminino] Página 14: Status 200
[Feminino] Página 15: Status 200
[Feminino] Página 16: Status 200
[Feminino] Página 17: Status 200
[Feminino] Página 18: Status 200
[Feminino] Página 19: Status 200
[Feminino] Página 20: Status 200
[Feminino] Página 21: Status 200
[Feminino] Página 22: Status 200
[Feminino] Página 23: Status 200
[Feminino] Página 24: Status 200
[Feminino] Página 25: Status 200
[Feminino] Página 26: Status 200
[Feminino] Página 27: Status 200
[Feminino] Página 28: Status 200
[Feminino] Página 29: Status 200
[Feminino] Pág